# Day 017 — Exercise 5: The Tool-Using Chatbot

**Goal:** Implement `run_tool_chatbot(tools, registry, system_prompt, model, max_turns)` — identical to `run_chatbot` from Day 15 but calling `tool_turn` instead of `chat_turn`.

In [ ]:
import ollama

In [ ]:
import ast, operator

def calculate(expression: str) -> str:
    """Evaluate a safe arithmetic expression and return the result as a string."""
    allowed = {
        ast.Add: operator.add, ast.Sub: operator.sub,
        ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.Pow: operator.pow, ast.Mod: operator.mod,
        ast.USub: operator.neg,
    }
    def _eval(node):
        if isinstance(node, ast.Constant):
            return node.value
        if isinstance(node, ast.BinOp):
            return allowed[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp):
            return allowed[type(node.op)](_eval(node.operand))
        raise ValueError(f"Unsafe expression: {ast.dump(node)}")
    result = _eval(ast.parse(expression, mode='eval').body)
    return str(result)

def get_weather(city: str) -> str:
    """Return a simulated current temperature for a city."""
    temperatures = {"london": "12\u00b0C", "tokyo": "22\u00b0C", "paris": "15\u00b0C",
                    "new york": "18\u00b0C", "sydney": "24\u00b0C"}
    temp = temperatures.get(city.lower(), "20\u00b0C")
    return f"The current temperature in {city} is {temp}."

TOOL_REGISTRY = {
    "calculate": calculate,
    "get_weather": get_weather,
}

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": (
                "Evaluate a mathematical expression and return the numeric result. "
                "Use this for any arithmetic including +, -, *, /, **, and %. "
                "Pass the expression as a string, e.g. '2847 * 193'."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "A Python math expression, e.g. '12 * 34'",
                    }
                },
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": (
                "Return the current temperature for a city. "
                "Use this when the user asks about current weather or temperature."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "City name, e.g. 'London' or 'Tokyo'",
                    }
                },
                "required": ["city"],
            },
        },
    },
]

In [ ]:
def has_tool_call(response: dict) -> bool:
    """Return True if the model's response contains at least one tool call."""
    return bool(response["message"].get("tool_calls"))

def extract_tool_call(response: dict) -> tuple[str, dict]:
    """Return (fn_name, fn_args) from the first tool call in the response."""
    call = response["message"]["tool_calls"][0]
    return call["function"]["name"], call["function"]["arguments"]

In [ ]:
def execute_tool(fn_name: str, fn_args: dict, registry: dict) -> str:
    """Look up fn_name in registry and call it with **fn_args."""
    if fn_name not in registry:
        raise KeyError(f"Unknown tool: {fn_name!r}")
    return registry[fn_name](**fn_args)

In [ ]:
def append_tool_result(
    messages: list[dict],
    assistant_msg: dict,
    tool_output: str,
) -> list[dict]:
    """Return new messages list with assistant tool-call msg and tool result appended."""
    tool_result = {"role": "tool", "content": tool_output}
    return messages + [assistant_msg, tool_result]

In [ ]:
def append_turn(history: list[dict], user_text: str, assistant_text: str) -> list[dict]:
    """Return a new history list with one user+assistant turn appended."""
    return history + [
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": assistant_text},
    ]

In [ ]:
import ollama

def tool_turn(
    history: list[dict],
    user_input: str,
    tools: list[dict],
    registry: dict,
    model: str = "llama3.2",
) -> tuple[str, list[dict]]:
    """Send user_input to the model and handle any tool call it makes."""
    messages = history + [{"role": "user", "content": user_input}]
    response = ollama.chat(model=model, messages=messages, tools=tools)

    if has_tool_call(response):
        fn_name, fn_args = extract_tool_call(response)
        tool_output = execute_tool(fn_name, fn_args, registry)
        messages = append_tool_result(messages, response["message"], tool_output)
        response = ollama.chat(model=model, messages=messages, tools=tools)

    reply = response["message"]["content"]
    return reply, append_turn(history, user_input, reply)

In [ ]:
def truncate_history(history: list[dict], max_turns: int = 10) -> list[dict]:
    """Keep the system prompt and the last max_turns*2 non-system messages."""
    if not history:
        return []
    if history[0]["role"] == "system":
        system, tail = [history[0]], history[1:]
    else:
        system, tail = [], history
    return system + tail[-(max_turns * 2):]

In [ ]:
def reset_history(history: list[dict]) -> list[dict]:
    """Return a new history containing only the system prompt (if present)."""
    if history and history[0]["role"] == "system":
        return [history[0]]
    return []

In [ ]:
def format_history(history: list[dict]) -> str:
    """Render conversation history as a readable transcript."""
    labels = {"user": "You", "assistant": "Bot"}
    lines = []
    for msg in history:
        if msg["role"] == "system":
            continue
        label = labels.get(msg["role"], msg["role"].capitalize())
        lines.append(f"{label}: {msg['content']}")
    return "\n".join(lines)

## Your Implementation

In [ ]:
SYSTEM_PROMPT = (
    "You are a helpful assistant with access to a calculator and weather lookup. "
    "Use the calculate tool for any arithmetic. "
    "Use the get_weather tool when asked about current temperature or weather."
)

def run_tool_chatbot(
    tools: list[dict] = TOOLS,
    registry: dict = TOOL_REGISTRY,
    system_prompt: str = SYSTEM_PROMPT,
    model: str = "llama3.2",
    max_turns: int = 10,
) -> None:
    """
    Run a multi-turn chatbot that can call tools.

    Identical to run_chatbot from Day 15 but uses tool_turn instead of chat_turn.
    """
    # TODO: init history with system prompt
    # TODO: print welcome banner
    # TODO: while True: read input, handle commands, call tool_turn
    pass

## Check Your Work

In [ ]:
import io, sys

def _run_checks():
    total = 5
    passed = 0

    # Check 1: all six functions are defined
    try:
        for name in ["has_tool_call", "extract_tool_call", "execute_tool",
                     "append_tool_result", "tool_turn", "run_tool_chatbot"]:
            assert name in globals(), f"{name} not defined"
        passed += 1; print("\u2705 Check 1: all six functions defined")
    except Exception as e:
        print(f"\u274c Check 1: missing function \u2014 {e}")

    # Check 2: tool_turn correctly uses calculate tool
    try:
        old = sys.stdout; sys.stdout = io.StringIO()
        reply, history = tool_turn([], "What is 2847 * 193?", TOOLS, TOOL_REGISTRY)
        sys.stdout = old
        assert "549471" in reply.replace(",", ""), f"expected '549471' in reply, got {reply!r}"
        passed += 1; print("\u2705 Check 2: calculate tool returns correct result")
    except Exception as e:
        sys.stdout = old
        print(f"\u274c Check 2: calculate tool \u2014 {e}")

    # Check 3: tool_turn correctly uses get_weather tool
    try:
        old = sys.stdout; sys.stdout = io.StringIO()
        reply2, history2 = tool_turn([], "What is the weather in Tokyo?", TOOLS, TOOL_REGISTRY)
        sys.stdout = old
        assert "22\u00b0C" in reply2 or "tokyo" in reply2.lower(), \
            f"expected temperature in reply, got {reply2!r}"
        passed += 1; print("\u2705 Check 3: get_weather tool returns temperature")
    except Exception as e:
        sys.stdout = old
        print(f"\u274c Check 3: get_weather tool \u2014 {e}")

    # Check 4: history management still works
    try:
        old = sys.stdout; sys.stdout = io.StringIO()
        _, h = tool_turn([], "What is 1 + 1?", TOOLS, TOOL_REGISTRY)
        sys.stdout = old
        trimmed = truncate_history(h, max_turns=10)
        reset = reset_history(h)
        assert len(trimmed) <= len(h)
        assert len(reset) == 0
        passed += 1; print("\u2705 Check 4: truncate_history and reset_history work with tool history")
    except Exception as e:
        sys.stdout = old
        print(f"\u274c Check 4: history management \u2014 {e}")

    # Check 5: execute_tool raises KeyError for unknown tool
    try:
        raised = False
        try:
            execute_tool("unknown_tool", {}, TOOL_REGISTRY)
        except KeyError:
            raised = True
        assert raised, "execute_tool should raise KeyError for unknown tool"
        passed += 1; print("\u2705 Check 5: execute_tool raises KeyError for unknown tool")
    except Exception as e:
        print(f"\u274c Check 5: KeyError \u2014 {e}")

    if passed == total:
        print("\U0001f389 Exercise complete!")
    print(f"\nScore: {passed}/{total}")

_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
SYSTEM_PROMPT = (
    "You are a helpful assistant with access to a calculator and weather lookup. "
    "Use the calculate tool for any arithmetic. "
    "Use the get_weather tool when asked about current temperature or weather."
)

def run_tool_chatbot(
    tools: list[dict] = TOOLS,
    registry: dict = TOOL_REGISTRY,
    system_prompt: str = SYSTEM_PROMPT,
    model: str = "llama3.2",
    max_turns: int = 10,
) -> None:
    history = [{"role": "system", "content": system_prompt}]
    print("Tool chatbot ready. Try asking for a calculation or weather.")
    print("Commands: /quit  /reset  /history")
    print("-" * 50)
    while True:
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye!")
            break
        if not user_input:
            continue
        if user_input.startswith("/"):
            if user_input == "/quit":
                print("Goodbye!")
                break
            elif user_input == "/reset":
                history = reset_history(history)
                print("Bot: Conversation reset.")
            elif user_input == "/history":
                transcript = format_history(history)
                print(transcript if transcript else "(no history yet)")
            else:
                print(f"Bot: Unknown command: {user_input}")
            continue
        reply, history = tool_turn(history, user_input, tools, registry, model)
        history = truncate_history(history, max_turns)
        print(f"Bot: {reply}")
```

</details>